# Train the LLM router on a Colab GPU

Fine-tunes `cross-encoder/ms-marco-MiniLM-L-6-v2` on your arena-built pairs. A free T4 is plenty — the full 3-epoch run over 15,000 pairs takes roughly 5–15 minutes.

**Before you start:**
1. `Runtime → Change runtime type → T4 GPU → Save`.
2. Push your latest local code to GitHub, or the clone below gets the old broken scripts:
   ```
   git add -A && git commit -m "Cross-platform training + dataset fixes" && git push
   ```

In [ ]:
# Should list an NVIDIA GPU. If not: Runtime > Change runtime type > GPU, then re-run.
!nvidia-smi

## 1. Get the code

In [ ]:
REPO = "/content/llm-router-project"
GITHUB_URL = "https://github.com/PolarDyth/llm-router-project.git"

import os
if not os.path.isdir(REPO):
    !git clone {GITHUB_URL} {REPO}

# No GitHub? Upload `training/train_router.py` and `device_utils.py` via the
# Files panel (folder icon on the left), then run:
#   !mkdir -p {REPO}/training
#   !mv /content/train_router.py {REPO}/training/
#   !mv /content/device_utils.py {REPO}/

## 2. Upload the dataset

Upload `data/router_training_pairs_scrubbed.csv` (~9 MB) from your Mac when prompted.

*Alternative:* upload `data/model_profiles.json` instead and rebuild here — `!python {REPO}/scripts/build_scrubbed_arena.py` (downloads the 330 MB arena dataset, a few minutes on Colab).

In [ ]:
import os, shutil
from google.colab import files

os.makedirs(f"{REPO}/data", exist_ok=True)
csv_path = f"{REPO}/data/router_training_pairs_scrubbed.csv"
if not os.path.exists(csv_path):
    uploaded = files.upload()  # pick router_training_pairs_scrubbed.csv
    for name in uploaded:
        shutil.move(name, f"{REPO}/data/{name}")
print("Dataset ready:", csv_path)

## 3. Install dependencies

Colab already ships a CUDA-enabled torch, which satisfies `torch>=2.0`, so pip leaves it alone and only installs/updates the rest.

In [ ]:
!pip install -q -r {REPO}/requirements.txt

## 4. Mount Google Drive (recommended)

Free Colab VMs are wiped when the session ends — train straight to Drive so the model survives.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

OUTPUT_DIR = "/content/drive/MyDrive/llm-router/dynamic_router_final"
# Prefer the ephemeral VM instead? Use:
# OUTPUT_DIR = f"{REPO}/models/dynamic_router/final"

## 5. Train

In [ ]:
!python {REPO}/training/train_router.py --dataset "{REPO}/data/router_training_pairs_scrubbed.csv" --output-dir "{OUTPUT_DIR}" --epochs 3 --device cuda

## 6. Get the model back (skip if you trained to Drive)

Zip and download, then unzip into `models/dynamic_router/final` in your local repo so `scripts/test_router.py` finds it.

In [ ]:
# from google.colab import files
# !zip -r /content/router_model.zip "{OUTPUT_DIR}"
# files.download("/content/router_model.zip")